# Module 5 & 4: Predictor Calculation & Predictor Stack Quality Analysis

## Overview
This Jupyter Notebook serves as the **executable specification and Proof of Concept (PoC)** for **Module 5 (Predictor Calculation)** and **Module 4 (Non-Parametric Predictor Stack Separability & Visual Diagnosis)** in the Luma Geospatial Engine (`luma-ge`). 

### Live Catalog Status
- **Total Predictors Registered**: 32
- **Spectral Indices Registered**: 19 (NDVI, EVI, SAVI, MSAVI, GNDVI, MCARI, RENDVI, MCARI1, MNDWI, NDMI, AWEIsh, AWEInsh, NDTI, ANDWI, NDBI, DBSI, BLFEI, MBI, EBBI)
- **Categories Covered**: Vegetation, Water, Built-up & Bare Soil, Topography, Radar Polarization

### Key Features Demonstrated:
1. **Sensor-Aware Predictor Catalog**: Dynamic filtering based on optical sensor (`LANDSAT` vs. `SENTINEL-2`) and temporal windows.
2. **Spectral Indices Engine**: Automated computation via `spyndex` and `luma-ge.predictor`.
3. **Terrain Predictors**: Elevation, Slope, Aspect, and Landform morphometry derived from DEMs (`NASADEM`, `ALOS`).
4. **SAR Radar Predictors**: Sentinel-1 C-band ($VV$, $VH$, ratio) and ALOS PALSAR / PALSAR-2 L-band ($HH$, $HV$, ratio).
5. **End-to-End Orchestration**: High-level execution via `PredictorCalculation.compute_predictors()`, maintaining complete contract compatibility with `luma-be`.
6. **Predictor Stack Quality & Separability Analysis (`sample_data_quality.py`)**:
   - **Non-Parametric Silhouette Score (`method='SILHOUETTE'`)**: Overall, per-class (`Negative_Ratio` outlier detection), and pairwise class separability.
   - **Predictor Set Comparison (`compare_predictor_sets`)**: Quantitative comparison of *Optical Only* vs. *Full Predictor Stack* vs. *Custom / RFECV Subset*.
   - **Dimensionality Reduction (`plot_tsne` & `plot_pca`)**: Interactive 2D/3D t-SNE cluster maps and PCA biplots with predictor loading arrows.
7. **Interactive Map Visualization**: Multi-layer rendering using `geemap`.

---
## 1. Environment Setup & Earth Engine Initialization
We initialize Google Earth Engine using `luma-ge`'s lazy initialization mechanism and import required geospatial libraries.

In [1]:
import ee
import geemap
import pandas as pd
from luma_ge.ee_config import ensure_ee_initialized
from luma_ge.data_acquisition import Reflectance_Data, final_Image
from luma_ge.predictor import (
    PredictorCalculation,
    predictor_catalog,
    SpectralCalculator,
    sensor_detector,
    terrain_calculator,
    distance_calculator
)
from luma_ge.sar import get_sar_data
from luma_ge.sample_data_quality import sample_quality, spectral_plotter

#Initialize Earth Engine

ensure_ee_initialized()
print("Google Earth Engine initialized successfully!")

2026-11-13 15:16:43,577 - luma_ge.ee_config - INFO - Earth Engine initialized successfully


Google Earth Engine initialized successfully!


---
## 2. Area of Interest (AOI), Time Window & Training Sample Placeholders
Replace the placeholders below with your target **AOI** and **Training Sample Data** (Shapefile, GeoDataFrame, or GEE FeatureCollection asset) when running your live verification.

In [13]:
#Define Area of Interest (AOI)
#Kambera, NTT
aoi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_AOI_Kambera_NTT')
#time period
start_date = '2025-01-01'
end_date = '2025-12-31'

#Training data
roi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_Data_Latih_Kambera')   
class_property = 'ID_Kelas' #custom define (class id)                          
class_name_property = 'KelasTupla' #custom define (class name)

---
## 3. Predictor Catalog & Sensor-Aware Availability Check
`PredictorCatalog` dynamically defines compatibility rules for all predictors.
- **Landsat-Specific**: e.g., `EBBI` (requires Thermal and SWIR).
- **Sentinel-2 Specific**: e.g., `RENDVI`, `MCARI` (require Red-Edge bands).
- **Radar**: Sentinel-1 ($VV$, $VH$ available from Oct 2014) vs. ALOS PALSAR ($HH$, $HV$).

In [3]:
predictor_service = PredictorCalculation()

# Query available predictors for Sentinel-2
s2_catalog = predictor_service.get_available_predictors(
    start_date=start_date,
    end_date=end_date,
    sensor='SENTINEL'
)

# Query available predictors for Landsat
landsat_catalog = predictor_service.get_available_predictors(
    start_date=start_date,
    end_date=end_date,
    sensor='LANDSAT'
)

# Access the inner predictors dictionaries
s2_preds = s2_catalog.get('predictors', {})
landsat_preds = landsat_catalog.get('predictors', {})

# Convert to a comparison DataFrame for inspection
df_comparison = pd.DataFrame([
    {
        "Predictor": name,
        "Category": meta.get('category', ''),
        "Sensor Target": meta.get('sensor_compatibility', meta.get('sensor', 'ALL')),
        "Sentinel-2": "Available" if s2_preds.get(name, {}).get('available') else f"Excluded: {s2_preds.get(name, {}).get('reason')}",
        "Landsat": "Available" if landsat_preds.get(name, {}).get('available') else f"Excluded: {landsat_preds.get(name, {}).get('reason')}"
    }
    for name, meta in s2_preds.items()
])

print(f"Total Predictors in Catalog: {len(df_comparison)}")
df_comparison

2026-11-13 15:16:43,634 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-13 15:16:44,317 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-13 15:16:44,318 - luma_ge.sar - INFO - get_sar_data initialized
2026-11-13 15:16:44,318 - luma_ge.predictor - INFO - Distance calculator initialized
2026-11-13 15:16:44,319 - luma_ge.predictor - INFO - Predictor Calculation initialized with terrain, spectral, SAR, and distance calculators


Total Predictors in Catalog: 32


,Predictor,Category,Sensor Target,Sentinel-2,Landsat
0,NDVI,Vegetasi,ALL,Available,Available
1,EVI,Vegetasi,ALL,Available,Available
2,SAVI,Vegetasi,ALL,Available,Available
3,MSAVI,Vegetasi,ALL,Available,Available
4,GNDVI,Vegetasi,ALL,Available,Available
5,MCARI,Vegetasi,SENTINEL,Available,Excluded: Requires Sentinel-2 Red-Edge band (RE1)
6,RENDVI,Vegetasi,SENTINEL,Available,Excluded: Requires Sentinel-2 Red-Edge bands (...
7,MCARI1,Vegetasi,ALL,Available,Available
8,MNDWI,Air & Kelembaban,ALL,Available,Available
9,NDMI,Air & Kelembaban,ALL,Available,Available


---
## 4. Optical Imagery Acquisition
We acquire standard cloud-masked, harmonized optical composites:
1. **Sentinel-2 Level-2A** via `ref_data.get_s2_optical_data()`
2. **Landsat 8 Level-2** via `ref_data.get_optical_data()`

In [4]:
ref_data = Reflectance_Data()
processor = final_Image()

# 1. Acquire Sentinel-2 Surface Reflectance Collection & Composite
print("Fetching Sentinel-2 collection...")
s2_col, s2_stats = ref_data.get_s2_optical_data(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    cloud_cover=40,
    compute_detailed_stats=False
)
s2_comp_res = processor.get_temporal_composite(s2_col, aoi, reducer='median')
s2_composite = s2_comp_res['image']
print("Sentinel-2 Bands:", s2_composite.bandNames().getInfo())

# 2. Acquire Landsat 8 Surface Reflectance Collection & Composite
print("\nFetching Landsat 8 collection...")
l8_col, l8_stats = ref_data.get_optical_data(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    optical_data='L8_SR',
    cloud_cover=40,
    compute_detailed_stats=False
)
l8_comp_res = processor.get_temporal_composite(l8_col, aoi, reducer='median')
l8_composite = l8_comp_res['image']
print("Landsat 8 Bands:", l8_composite.bandNames().getInfo())

2026-11-13 15:16:44,357 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-11-13 15:16:44,358 - final_Image - INFO - final_Image creation initialized.
2026-11-13 15:16:44,359 - Reflectance_Data - INFO - Date range: 2025-01-01 to 2025-12-31
2026-11-13 15:16:44,361 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 40%
2026-11-13 15:16:44,361 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.7
2026-11-13 15:16:44,362 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-11-13 15:16:44,363 - Reflectance_Data - INFO - Using Sentinel-2 Level 2A SR data.
2026-11-13 15:16:44,366 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-13 15:16:44,367 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)


Fetching Sentinel-2 collection...


2026-11-13 15:16:45,052 - final_Image - INFO - Creating median composite from 69 images
2026-11-13 15:16:45,054 - final_Image - INFO - Composite clipped to AOI
2026-11-13 15:16:50,284 - final_Image - INFO - Composite created from 2025-03-01 to 2025-12-13
2026-11-13 15:16:55,946 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-11-13 15:16:55,947 - Reflectance_Data - INFO - Date range: 2025-01-01 to 2025-12-31
2026-11-13 15:16:55,948 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2026-11-13 15:16:55,949 - Reflectance_Data - INFO - detailed statistics will not be computed
2026-11-13 15:16:55,950 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-13 15:16:55,952 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)


Sentinel-2 Bands: ['BLUE', 'GREEN', 'RED', 'RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'NIR', 'RED_EDGE4', 'SWIR1', 'SWIR2']

Fetching Landsat 8 collection...


2026-11-13 15:17:00,174 - final_Image - INFO - Creating median composite from 11 images
2026-11-13 15:17:00,176 - final_Image - INFO - Composite clipped to AOI
2026-11-13 15:17:02,126 - final_Image - INFO - Composite created from 2025-02-16 to 2025-10-30


Landsat 8 Bands: ['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']


---
## 5. Individual Predictor Components
Standalone computation of each predictor component:
1. **Spectral Indices** (`SpectralCalculator`)
2. **Terrain Predictors** (`terrain_calculator`)
3. **Radar SAR Polarizations** (`sar.py`)
4. **Distance & Accessibility Predictors** (`distance_calculator`)

### 5.1 Spectral Indices Calculation
Computing standard indices alongside sensor-specific indices:
- `RENDVI` and `MCARI` computed on Sentinel-2
- `DBSI` and `MBI` computed on Landsat-8

In [5]:
# Sentinel-2 Indices (Testing Red-Edge index: RENDVI, MCARI)
s2_calc = SpectralCalculator()
s2_indices = s2_calc.calculate_indices(
    image=s2_composite,
    index_list=['NDVI', 'EVI', 'MNDWI', 'RENDVI', 'MCARI'],
    sensor='SENTINEL'
)
print("Sentinel-2 Computed Indices:", s2_indices.bandNames().getInfo())

# Landsat-8 Indices (Testing Soil and Water: DBSI, MBI, MNDWI)
l8_calc = SpectralCalculator()
l8_indices = l8_calc.calculate_indices(
    image=l8_composite,
    index_list=['NDVI', 'MNDWI', 'DBSI', 'MBI'],
    sensor='LANDSAT'
)
print("Landsat-8 Computed Indices:", l8_indices.bandNames().getInfo())

2026-11-13 15:18:22,958 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-13 15:18:24,290 - luma_ge.predictor - INFO - Retrieved default coefficients for EVI: {'g': 2.5, 'C1': 6.0, 'C2': 7.5, 'L': 1.0}
2026-11-13 15:18:25,840 - luma_ge.predictor - INFO - SpectralCalculator initialized


Sentinel-2 Computed Indices: ['NDVI', 'EVI', 'MNDWI', 'RENDVI', 'MCARI']
Landsat-8 Computed Indices: ['NDVI', 'MNDWI', 'DBSI', 'MBI']


### 5.2 Terrain Predictors (Topography & Geomorphology)
Using `terrain_calculator` to extract Elevation, Slope, Aspect, and morphological Landform classes.

In [6]:
tc = terrain_calculator()

# 1. Elevation (from NASADEM)
dem_elevation = tc.calculate_elevation(aoi)
print("Elevation Bands:", dem_elevation.bandNames().getInfo())

# 2. Morphological Landforms (Penumbral, Ridges, Valleys, Plains)
landform_image = tc.get_landform(aoi)
print("Landform Bands:", landform_image.bandNames().getInfo())

2026-11-13 15:18:28,084 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-13 15:18:28,087 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-13 15:18:28,088 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-13 15:18:29,497 - luma_ge.predictor - INFO - Retrieving landform layer from CSP/ERGo/1_0/Global/ALOS_landforms...
2026-11-13 15:18:29,499 - luma_ge.predictor - INFO - Successfully retrieved landform layer using ALOS source


Elevation Bands: ['elevation']
Landform Bands: ['landform']


### 5.3 Radar SAR Backscatter (Sentinel-1 & ALOS PALSAR)
Using `get_sar_data()` to extract C-band ($VV$, $VH$) and L-band ($HH$, $HV$).

In [7]:
sar_service = get_sar_data()

# Sentinel-1 C-Band
s1_sar = sar_service.get_sentinel1(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    orbit_pass='BOTH',
    reducer='median',
    include_ratio=True
)
print("Sentinel-1 SAR Bands:", s1_sar.bandNames().getInfo())

# ALOS PALSAR L-Band (with automatic gap fallback)
palsar_sar = sar_service.get_palsar(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    allow_nearest_year_fallback=True
)
print("ALOS PALSAR SAR Bands:", palsar_sar.bandNames().getInfo())

2026-11-13 15:18:30,971 - luma_ge.sar - INFO - get_sar_data initialized
2026-11-13 15:18:31,414 - luma_ge.sar - INFO - Sentinel-1: 83 scenes for 2025-01-01 to 2025-12-31 (BOTH)
2026-11-13 15:18:32,209 - luma_ge.sar - WARNING - No PALSAR yearly mosaic directly available for 2025-2025 (e.g. 2011-2014 mission gap or out-of-range). Falling back to nearest year: 2024


Sentinel-1 SAR Bands: ['S1_VV', 'S1_VH', 'S1_VV_VH']


2026-11-13 15:18:33,704 - luma_ge.sar - INFO - PALSAR: 1 yearly mosaic scenes retrieved for years [2024]


ALOS PALSAR SAR Bands: ['PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV']


### 5.4 Distance & Accessibility Predictors (GRIP4 Main Roads, S2COAST-2023, HRSL)
Using `distance_calculator` with Earth Engine's linear-time `fastDistanceTransform()` clipped to the user-defined `aoi`:
- `dist_roads`: On-the-fly from Global Roads Inventory Project (`GRIP4`, main roads `GP_RTP` in `[1, 2, 3]`) or pre-computed asset (`road_source='precomputed'`)
- `dist_coast`: S2COAST-2023 (`projects/sat-io/open-datasets/S2COAST-2023`) clipped to `aoi`
- `dist_settlement`: High Resolution Settlement Layer (`HRSL`)

In [8]:
dc = distance_calculator()

# Compute all 3 distance predictors on-the-fly clipped to AOI
distance_stack = dc.calculate_distance_metrics(
    aoi=aoi,
    max_dist=50000,
    in_meters=True,
    include_roads=True,
    include_coast=True,
    include_settlement=True,
    road_source="grip4",
    main_roads_only=True
)
print("Distance Predictor Bands:", distance_stack.bandNames().getInfo())

2026-11-13 15:18:34,448 - luma_ge.predictor - INFO - Distance calculator initialized
2026-11-13 15:18:34,450 - luma_ge.predictor - INFO - Calculating fast distance metrics (max_dist=50000, in_meters=True, road_source=grip4, main_roads_only=True)...
2026-11-13 15:18:34,455 - luma_ge.predictor - INFO - Successfully constructed fast distance predictor stack


Distance Predictor Bands: ['dist_roads', 'dist_coast', 'dist_settlement']


---
## 6. End-to-End Pipeline Orchestration: `compute_predictors()`
This is the primary method called by the backend service (`lulc_map.py` and `thematic.py`).

Notice that the configuration dynamically pulls **all registered spectral indices directly from `PredictorCatalog`**, so any new index added to `predictor.py` is automatically computed and tested here!

In [22]:
# Dynamically pull all registered spectral indices from the catalog
all_indices = [
    name for name, meta in predictor_catalog.get_all_catalog().items()
    if meta.get("type") == "spectral"
]

print(f"Testing with all {len(all_indices)} registered spectral indices:")
print(all_indices)

# Define full predictor configuration (matching backend payload schema)
predictor_config = {
    "individual_predictors": {
        "elevation": True,
        "slope": True,
        "aspect": True,
        "landform": True,
        "spectral_indices": all_indices,
        "sentinel1": True,
        "dist_roads": True,
        "palsar": True
    }
}

orchestrator = PredictorCalculation()

# Execute on Sentinel-2
print("\n=== Running compute_predictors on Sentinel-2 ===")
s2_result = orchestrator.compute_predictors(
    composite=s2_composite,
    aoi=aoi,
    predictor_config=predictor_config,
    collection=s2_col
)

print(f"Status: Success={s2_result['success']}")
print(f"Detected Sensor: {s2_result['sensor']}")
print(f"Total Output Bands: {s2_result['total_bands']}")
print("Output Band Names:")
print(s2_result['stacked_predictors'].bandNames().getInfo())

if s2_result['warnings']:
    print("\nExpected Warnings (graceful skipping of Landsat-only indices):")
    for w in s2_result['warnings']:
        print(f"  - {w}")

2026-11-13 15:32:33,315 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-11-13 15:32:33,316 - luma_ge.predictor - INFO - SpectralCalculator initialized
2026-11-13 15:32:33,318 - luma_ge.sar - INFO - get_sar_data initialized
2026-11-13 15:32:33,319 - luma_ge.predictor - INFO - Distance calculator initialized
2026-11-13 15:32:33,320 - luma_ge.predictor - INFO - Predictor Calculation initialized with terrain, spectral, SAR, and distance calculators


Testing with all 19 registered spectral indices:
['NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI', 'RENDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'EBBI']

=== Running compute_predictors on Sentinel-2 ===


2026-11-13 15:32:34,632 - luma_ge.predictor - INFO - Prerequisite validation completed: valid=True
2026-11-13 15:32:37,076 - luma_ge.predictor - INFO - Predictor computation utilizing detected sensor: SENTINEL
2026-11-13 15:32:39,384 - luma_ge.predictor - INFO - Added 10 original multispectral bands
2026-11-13 15:32:39,385 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-13 15:32:39,387 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-13 15:32:39,388 - luma_ge.predictor - INFO - Successfully computed elevation
2026-11-13 15:32:39,389 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-11-13 15:32:39,390 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-13 15:32:39,392 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-13 15:32:39,394 - luma_ge.predictor - INFO - Successfully calculated slope layer u

Status: Success=True
Detected Sensor: SENTINEL
Total Output Bands: 39
Output Band Names:
['BLUE', 'GREEN', 'RED', 'RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'NIR', 'RED_EDGE4', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'aspect', 'landform', 'NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI', 'RENDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'S1_VV', 'S1_VH', 'S1_VV_VH', 'PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV', 'dist_roads']


In [10]:
# Execute on Landsat 8
print("\n=== Running compute_predictors on Landsat-8 ===")
l8_result = orchestrator.compute_predictors(
    composite=l8_composite,
    aoi=aoi,
    predictor_config=predictor_config,
    collection=l8_col
)

print(f"Status: Success={l8_result['success']}")
print(f"Detected Sensor: {l8_result['sensor']}")
print(f"Total Output Bands: {l8_result['total_bands']}")
print("Output Band Names:")
print(l8_result['stacked_predictors'].bandNames().getInfo())

if l8_result['warnings']:
    print("\nExpected Warnings (graceful skipping of Sentinel-only indices):")
    for w in l8_result['warnings']:
        print(f"  - {w}")


=== Running compute_predictors on Landsat-8 ===


2026-11-13 15:19:31,085 - luma_ge.predictor - INFO - Prerequisite validation completed: valid=True
2026-11-13 15:19:36,165 - luma_ge.predictor - INFO - Predictor computation utilizing detected sensor: LANDSAT
2026-11-13 15:19:37,754 - luma_ge.predictor - INFO - Added 7 original multispectral bands
2026-11-13 15:19:37,755 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-13 15:19:37,757 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-13 15:19:37,758 - luma_ge.predictor - INFO - Successfully computed elevation
2026-11-13 15:19:37,759 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-11-13 15:19:37,761 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-11-13 15:19:37,763 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-11-13 15:19:37,765 - luma_ge.predictor - INFO - Successfully calculated slope layer usi

Status: Success=True
Detected Sensor: LANDSAT
Total Output Bands: 33
Output Band Names:
['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'elevation', 'slope', 'aspect', 'landform', 'NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'S1_VV', 'S1_VH', 'S1_VV_VH', 'PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV']


---
## 7. Predictor Stack & Sample Quality Analysis (`sample_data_quality.py`)
Once the full predictor stack (`s2_result['stacked_predictors']` or `l8_result['stacked_predictors']`) is computed, we evaluate how well the selected predictors separate our training classes using:
1. **Non-Parametric Silhouette Score (`method='SILHOUETTE'`)**: Evaluates cluster cohesion and separation in standardized predictor space without assuming Gaussian normality.
2. **Predictor Subset Comparison (`compare_predictor_sets`)**: Compares *Optical Only* vs. *Optical + Indices* vs. *Full Stack* vs. *RFECV / Custom Subset*.
3. **Dimensionality Reduction (`plot_tsne` & `plot_pca`)**: Visualizes various predictor dimensions in 2D/3D space to inspect overlapping classes, multi-modal clusters, and predictor loading arrows.

### 7.1 Extract Predictor Values at Training Sample Locations
Select which stacked image to analyze (`target_stacked_image`) and extract pixel values for `labeled_roi`.

In [14]:
# Choose which stacked predictor image to evaluate (Sentinel-2 or Landsat-8)
target_stacked_image = s2_result['stacked_predictors']  # or l8_result['stacked_predictors']
sampling_scale = 30  # Resolution in meters (e.g., 10 or 30)
max_pixels_per_class = 5000

if roi is None:
    raise ValueError(
        "Please set `labeled_roi` (and `class_property` / `class_name_property`) in Section 2 "
        "before running Section 7."
    )

analyzer = sample_quality(
    training_data=roi,
    image=target_stacked_image,
    class_property=class_property,
    region=aoi,
    class_name_property=class_name_property
)

print("Extracting predictor stack values at sample locations on GEE...")
pixel_extract = analyzer.extract_spectral_values(
    scale=sampling_scale,
    max_pixels_per_class=max_pixels_per_class
)
print(f"Extracted {len(pixel_extract)} sample pixels across {len(analyzer.band_names)} predictor bands.")
pixel_extract.head()

Extracting predictor stack values at sample locations on GEE...
Extracted spectral values for 459 samples across 9 classes
Extracted 459 sample pixels across 38 predictor bands.


,ANDWI,AWEInsh,AWEIsh,BLFEI,BLUE,DBSI,EVI,GNDVI,GREEN,ID_Kelas,...,S1_VH,S1_VV,S1_VV_VH,SAVI,SWIR1,SWIR2,aspect,elevation,landform,slope
0,0.119718,1236.191489,480.691489,-0.012520,552.0,-0.065588,0.177757,-0.158630,622.0,9,...,-23.502647,-17.333605,6.169043,-0.062840,520.0,310.0,270.000000,12,41,0.940822
1,0.366695,1940.925532,1187.308511,0.238909,549.0,-0.068866,0.579493,-0.402991,686.0,9,...,-27.128654,-18.747482,8.381172,-0.464154,239.0,182.0,106.472420,21,41,9.718568
2,-0.656133,-4022.379630,-7595.546296,-0.489385,391.0,-0.238536,2.229428,0.714421,683.0,9,...,-15.415924,-8.988696,6.427228,1.168184,2203.0,1103.0,71.809387,19,42,2.968361
3,0.224676,1584.750000,922.807018,0.039090,599.0,-0.003457,-0.060303,-0.320240,684.0,9,...,-25.608984,-20.353949,5.255034,-0.276424,448.0,275.0,225.408951,6,31,1.320899
4,-0.515171,-1042.813953,-3588.482558,-0.364305,334.0,-0.442452,2.820433,0.618448,537.0,9,...,-19.701965,-15.047121,4.654844,1.101933,1026.0,488.0,270.000000,7,34,1.881027


### 7.2 Per-Class & Pairwise Non-Parametric Silhouette Analysis
- **Per-Class Table**: Inspect `Mean_Silhouette` and `Negative_Ratio` (fraction of samples closer to another class than their own, indicating mislabeled polygons or class overlap).
- **Pairwise Table**: Inspect which specific class pairs have low separation (`Silhouette_Score < 0.25`).

In [15]:
# 1. Per-Class Silhouette Summary
class_silhouette_df = analyzer.get_class_silhouette_df(pixel_extract)
print("=== Per-Class Silhouette Quality Summary ===")
display(class_silhouette_df)

# 2. Pairwise Class Separability using Non-Parametric Silhouette Score
pairwise_silhouette_df = analyzer.get_separability_df(pixel_extract, method='SILHOUETTE')
print("\n=== Pairwise Class Separability (Silhouette Score) ===")
display(pairwise_silhouette_df)

# 3. Overall Separability Summary Count (Good / Weak / Poor pairs)
sep_summary_df = analyzer.sum_separability(pixel_extract, method='SILHOUETTE')
print("\n=== Silhouette Separability Summary ===")
display(sep_summary_df)

=== Per-Class Silhouette Quality Summary ===


,Class_ID,Class_Name,Sample_Count,Mean_Silhouette,Std_Silhouette,Min_Silhouette,Negative_Ratio,Separability_Level
0,8,Lahan Terbuka,72,-0.087,0.202,-0.574,0.653,🔴 Poor Separability (Silhouette < 0.25)
1,7,Lahan Terbangun,37,-0.003,0.139,-0.387,0.405,🔴 Poor Separability (Silhouette < 0.25)
2,5,Pertanian Lahan Kering,31,-0.000,0.208,-0.481,0.387,🔴 Poor Separability (Silhouette < 0.25)
3,4,Padang Rumput,78,0.178,0.117,-0.221,0.077,🔴 Poor Separability (Silhouette < 0.25)
4,1,Hutan Lahan Kering,92,0.210,0.124,-0.177,0.087,🔴 Poor Separability (Silhouette < 0.25)
5,3,Kebun Campur,33,0.237,0.152,-0.331,0.091,🔴 Poor Separability (Silhouette < 0.25)
6,6,Sawah,28,0.304,0.164,-0.261,0.071,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...
7,9,Badan Air,49,0.359,0.348,-0.732,0.143,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...
8,2,Hutan Mangrove,39,0.384,0.083,0.153,0.000,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...



=== Pairwise Class Separability (Silhouette Score) ===


,Class1_ID,Class1_Name,Class2_ID,Class2_Name,Silhouette_Score,Separability_Level
0,5,Pertanian Lahan Kering,8,Lahan Terbuka,0.082,🔴 Poor Separability (Silhouette < 0.25)
1,7,Lahan Terbangun,8,Lahan Terbuka,0.139,🔴 Poor Separability (Silhouette < 0.25)
2,4,Padang Rumput,8,Lahan Terbuka,0.143,🔴 Poor Separability (Silhouette < 0.25)
3,5,Pertanian Lahan Kering,7,Lahan Terbangun,0.161,🔴 Poor Separability (Silhouette < 0.25)
4,4,Padang Rumput,5,Pertanian Lahan Kering,0.177,🔴 Poor Separability (Silhouette < 0.25)
5,1,Hutan Lahan Kering,3,Kebun Campur,0.232,🔴 Poor Separability (Silhouette < 0.25)
6,5,Pertanian Lahan Kering,6,Sawah,0.269,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...
7,3,Kebun Campur,5,Pertanian Lahan Kering,0.276,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...
8,6,Sawah,7,Lahan Terbangun,0.278,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...
9,6,Sawah,8,Lahan Terbuka,0.289,🟡 Weak/Marginal Separability (0.25 ≤ Silhouett...



=== Silhouette Separability Summary ===


,Total Pairs,Good Separability Pairs,Weak Separability Pairs,Poor Separability Pairs
0,36,13,17,6


### 7.3 Compare Candidate Predictor Subsets (`compare_predictor_sets`)
Compare different combinations of predictors (e.g., raw optical bands vs. full predictor stack vs. your RFECV-selected subset) to verify whether adding terrain, radar, or specific indices improves class separation.

In [16]:
available_stack_bands = [b for b in analyzer.band_names if b in pixel_extract.columns]

# Define candidate predictor sets to compare (customize `rfecv_selected_bands` with your RFECV output)
optical_only_bands = [
    b for b in ['AEROSOL', 'BLUE', 'GREEN', 'RED', 'RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'NIR', 'RED_EDGE4', 'SWIR1', 'SWIR2']
    if b in available_stack_bands
]
optical_plus_indices = [
    b for b in available_stack_bands
    if b not in ['elevation', 'slope', 'aspect', 'landform', 'S1_VV', 'S1_VH', 'S1_VV_VH', 'PALSAR_HH', 'PALSAR_HV', 'PALSAR_HH_HV']
]

# PLACEHOLDER: Replace with your RFECV selected band list if available
rfecv_selected_bands = [
    b for b in ['RED', 'NIR', 'SWIR1', 'NDVI', 'MNDWI', 'RENDVI', 'DBSI', 'elevation', 'slope', 'S1_VV', 'S1_VH']
    if b in available_stack_bands
]

predictor_sets_to_compare = {
    "1. Optical Bands Only": optical_only_bands,
    "2. Optical + Spectral Indices": optical_plus_indices,
    "3. Full Predictor Stack (All Bands)": available_stack_bands,
    "4. RFECV / Custom Selected Subset": rfecv_selected_bands,
}

comparison_df = analyzer.compare_predictor_sets(
    df=pixel_extract,
    predictor_sets=predictor_sets_to_compare,
    max_samples=5000
)
print("=== Predictor Subset Quality Comparison ===")
display(comparison_df)

=== Predictor Subset Quality Comparison ===


,Predictor_Set,Num_Bands,Overall_Silhouette,Mean_Pairwise_Silhouette,Min_Pairwise_Silhouette,Worst_Class_Pair,Negative_Sample_Ratio,Bands
0,4. RFECV / Custom Selected Subset,11,0.238,0.483,0.122,Pertanian Lahan Kering vs Lahan Terbuka,0.186,"RED, NIR, SWIR1, NDVI, MNDWI, RENDVI, DBSI, el..."
1,2. Optical + Spectral Indices,28,0.207,0.523,0.085,Pertanian Lahan Kering vs Lahan Terbuka,0.308,"BLUE, GREEN, RED, RED_EDGE1, RED_EDGE2, RED_ED..."
2,3. Full Predictor Stack (All Bands),38,0.165,0.406,0.082,Pertanian Lahan Kering vs Lahan Terbuka,0.213,"BLUE, GREEN, RED, RED_EDGE1, RED_EDGE2, RED_ED..."
3,1. Optical Bands Only,10,0.151,0.491,0.024,Lahan Terbangun vs Lahan Terbuka,0.346,"BLUE, GREEN, RED, RED_EDGE1, RED_EDGE2, RED_ED..."


### 7.4 Dimensionality Reduction & Visual Diagnosis (`plot_tsne` & `plot_pca`)
- **`plot_tsne()`**: Non-linearly projects all selected predictors into 2D/3D space to reveal cluster separation, multi-modal subclasses, and isolated mislabeled points.
- **`plot_pca()`**: Projects predictors onto the top Principal Components and overlays **predictor loading vectors** showing which bands pull which classes apart.

In [17]:
plotter = spectral_plotter(analyzer)

# Choose which band subset to visualize (e.g., available_stack_bands or rfecv_selected_bands)
bands_to_visualize = available_stack_bands

# 1. Interactive 2D t-SNE Projection
fig_tsne_2d = plotter.plot_tsne(
    df=pixel_extract,
    bands=bands_to_visualize,
    n_components=2,
    perplexity=30.0,
    max_samples=3000,
    interactive=True
)
fig_tsne_2d.show()

In [ ]:
# 2. Interactive 3D t-SNE Projection (Optional exploratory view)
fig_tsne_3d = plotter.plot_tsne(
    df=pixel_extract,
    bands=bands_to_visualize,
    n_components=3,
    perplexity=30.0,
    max_samples=3000,
    interactive=True
)
fig_tsne_3d.show()

In [ ]:
# 3. Interactive 2D PCA Biplot with Top Predictor Loading Arrows
fig_pca_2d = plotter.plot_pca(
    df=pixel_extract,
    bands=bands_to_visualize,
    n_components=2,
    max_samples=5000,
    show_loadings=True,
    top_n_loadings=8,
    interactive=True
)
fig_pca_2d.show()

---
## 8. Interactive Multi-Layer Visualization (`geemap`)
Visual inspection of predictors over the AOI to verify radiometric quality, spatial resolution alignment, and edge contrasts.

In [23]:
# Initialize Geemap canvas centered on AOI
Map = geemap.Map()
Map.centerObject(aoi, 11)

# 1. Base Optical Imagery (Natural Color & False Color NIR)
s2_stacked = s2_result['stacked_predictors']
Map.addLayer(
    s2_stacked,
    {'bands': ['RED', 'GREEN', 'BLUE'], 'min': 0, 'max': 3000},
    '1. S2 True Color (RGB)'
)
Map.addLayer(
    s2_stacked,
    {'bands': ['NIR', 'RED', 'GREEN'], 'min': 0, 'max': 4500},
    '2. S2 False Color Infrared (CIR)'
)

# 3. Spectral Indices
# NDVI (Vegetation Greenness)
ndvi_vis = {'bands': ['NDVI'], 'min': 0.0, 'max': 0.85, 'palette': ['blue', 'white', 'green']}
Map.addLayer(s2_stacked, ndvi_vis, '3. NDVI (Vegetation)')

# MNDWI (Water / Moisture)
mndwi_vis = {'bands': ['MNDWI'], 'min': -0.5, 'max': 0.5, 'palette': ['brown', 'yellow', 'cyan', 'blue']}
Map.addLayer(s2_stacked, mndwi_vis, '4. MNDWI (Water/Moisture)')

# RENDVI (Sentinel-2 Red Edge NDVI)
rendvi_vis = {'bands': ['RENDVI'], 'min': 0.0, 'max': 0.7, 'palette': ['red', 'yellow', 'darkgreen']}
Map.addLayer(s2_stacked, rendvi_vis, '5. RENDVI (Red Edge)')

# 4. Topography & Terrain
elev_vis = {'bands': ['elevation'], 'min': 0, 'max': 150, 'palette': ['darkblue', 'teal', 'lightgreen', 'brown', 'white']}
Map.addLayer(s2_stacked, elev_vis, '6. NASADEM Elevation')

landform_vis = {'bands': ['landform'], 'min': 11, 'max': 42, 'palette': ['141414', '383838', '808080', 'EBEBEB']}
Map.addLayer(s2_stacked, landform_vis, '7. Morphological Landforms')

# 5. Radar Backscatter (Sentinel-1 SAR)
s1_sar_vis = {'bands': ['S1_VV', 'S1_VH', 'S1_VV_VH'], 'min': [-18, -25, 0.2], 'max': [-4, -10, 0.9]}
Map.addLayer(s2_stacked, s1_sar_vis, '8. Sentinel-1 SAR (VV/VH/Ratio)')

dist_road_vis = {'bands': ['dist_roads'], 'min': 0, 'max': 5000}
Map.addLayer(s2_stacked, dist_road_vis, '9. Distance From Main Roads')

# Add AOI boundary outline
aoi_outline = ee.Image().paint(ee.FeatureCollection([ee.Feature(aoi)]), color=1, width=2)
Map.addLayer(aoi_outline, {'palette': ['FF0000']}, 'AOI Boundary')

# Overlay training sample polygons/points if loaded
if roi is not None:
    Map.addLayer(roi, {}, 'Training Samples (ROI)')

# Render the interactive map
Map

Map(center=[-9.704356522667307, 120.2721675324136], controls=(WidgetControl(options=['position', 'transparent_…

---
## 9. Technical Specification & Contract for Backend & Frontend

### 9.1 Backend Implementation Contract (`luma-be`)
When the frontend submits the predictor configuration, the payload structured below must be stored in `luma.predictor_config` and passed into `compute_predictors()`:

```json
{
  "individual_predictors": {
    "elevation": true,
    "slope": true,
    "aspect": true,
    "landform": true,
    "spectral_indices": ['NDVI', 'EVI', 'SAVI', 'MSAVI', 'GNDVI', 'MCARI', 'RENDVI', 'MCARI1', 'MNDWI', 'NDMI', 'AWEIsh', 'AWEInsh', 'NDTI', 'ANDWI', 'NDBI', 'DBSI', 'BLFEI', 'MBI', 'EBBI'],
    "sentinel1": true,
    "palsar": false
  }
}
```

> **Important**: The backend whitelist in `application/logic/luma/predictor.py` (`set_predictor_config()`) should accept all predictors defined in `PredictorCatalog.get_all_catalog().keys()`.

### 9.2 Frontend Layer Styling Guide (`luma-fe`)
Recommended layer visualization palettes for frontend map rendering:

| Predictor Type | Band Name | Min | Max | Recommended Palette |
| :--- | :--- | :--- | :--- | :--- |
| **Vegetation** | `NDVI`, `EVI`, `GNDVI` | `0.0` | `0.85` | `['#0000FF', '#FFFFFF', '#00AA00']` |
| **Red Edge** | `RENDVI`, `MCARI` | `0.0` | `0.70` | `['#FF0000', '#FFFF00', '#006400']` |
| **Water** | `MNDWI`, `NDMI`, `AWEIsh` | `-0.5` | `0.50` | `['#8B4513', '#FFFFE0', '#00FFFF', '#0000FF']` |
| **Built-up / Soil** | `EBBI`, `DBSI`, `BLFEI`, `MBI`, `BI` | `-0.5` | `0.80` | `['#228B22', '#FFFF00', '#FF0000', '#800080']` |
| **Elevation** | `elevation` | `0` | `2000` | `['#00008B', '#008080', '#90EE90', '#8B4513', '#FFFFFF']` |
| **SAR Radar** | `S1_VV` / `S1_VH` | `-25 dB` | `0 dB` | Grayscale `['#000000', '#FFFFFF']` |